# EDA & Data Cleansing — MBG Sentiment Dataset

This notebook documents the exploratory data analysis and cleaning pipeline
for the **Makan Bergizi Gratis (MBG)** sentiment analysis project.

The raw dataset is built from **four** sources:
- **X** (Twitter) — scraped via snscrape & twikit
- **Kaggle** — third-party tweet collection
- **YouTube** — comments scraped from relevant videos
- **Instagram** — comments exported from public posts

Goal: produce a clean, deduplicated text corpus ready for SVM-based sentiment classification.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Raw Merged Dataset

In [ ]:
df_raw = pd.read_csv("../../dataset/raw/merged_raw.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head(8)

## 2. Source Distribution

In [ ]:
counts = df_raw["source"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {"x": "#1DA1F2", "kaggle": "#20BEFF", "youtube": "#FF0000", "instagram": "#E1306C"}
axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
axes[0].set_title("Rows per Source")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=[colors[s] for s in counts.index], startangle=90)
axes[1].set_title("Source Proportion")
plt.tight_layout()
plt.show()

## 3. Check Raw Text — Common Issues

Before cleaning we need to understand what noise exists:
- URLs (`https://t.co/...`, `youtube.com/...`)
- Mentions (`@username`)
- Emojis 🟣😂
- HTML entities (`&amp;`)
- Smart quotes / curly quotes
- Hashtag spam
- Empty or 1-word rows

In [ ]:
import re

def count_issues(series):
    return {
        "has_url": series.str.contains(r"https?://\S+", regex=True).sum(),
        "has_mention": series.str.contains(r"@\w+", regex=True).sum(),
        "has_entity": series.str.contains(r"&[a-z]+;", regex=True).sum(),
        "avg_chars": series.str.len().mean(),
        "avg_words": series.str.split().str.len().mean(),
        "empty": (series.str.strip() == "").sum(),
    }

issues = {}
for src in df_raw["source"].unique():
    mask = df_raw["source"] == src
    issues[src] = count_issues(df_raw.loc[mask, "text"])

issues_df = pd.DataFrame(issues).T.astype(int)
issues_df["total"] = df_raw["source"].value_counts()
issues_df

In [ ]:
# Show a few worst-case raw examples (long, URLs, emojis, etc.)
df_raw.loc[df_raw["text"].str.len().sort_values(ascending=False).index[:10],
          ["source", "text"]].style.set_properties(subset=["text"], **{"width": "600px"})

## 4. Text Length Distribution (Raw vs. Cleaned)

In [ ]:
from data_clean import clean_text

df_raw["cleaned"] = df_raw["text"].astype(str).apply(clean_text)
df_raw["raw_len"] = df_raw["text"].str.len()
df_raw["clean_len"] = df_raw["cleaned"].str.len()
df_raw["raw_words"] = df_raw["text"].str.split().str.len()
df_raw["clean_words"] = df_raw["cleaned"].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df_raw["raw_len"].clip(upper=500), bins=50, color="coral", alpha=0.7, label="Raw")
axes[0, 0].hist(df_raw["clean_len"].clip(upper=500), bins=50, color="steelblue", alpha=0.7, label="Cleaned")
axes[0, 0].set_title("Character Length (clipped at 500)")
axes[0, 0].legend()

axes[0, 1].hist(df_raw["raw_words"].clip(upper=60), bins=40, color="coral", alpha=0.7, label="Raw")
axes[0, 1].hist(df_raw["clean_words"].clip(upper=60), bins=40, color="steelblue", alpha=0.7, label="Cleaned")
axes[0, 1].set_title("Word Count (clipped at 60)")
axes[0, 1].legend()

axes[1, 0].hist(df_raw["raw_len"] - df_raw["clean_len"], bins=40, color="green", alpha=0.7)
axes[1, 0].set_title("Characters Removed per Row")
axes[1, 0].axvline(x=0, color="red", linestyle="--")

for i, src in enumerate(df_raw["source"].unique()):
    subset = df_raw[df_raw["source"] == src]
    axes[1, 1].barh(src, subset["clean_words"].mean(), color=colors.get(src, "gray"))
axes[1, 1].set_title("Average Words per Source (after cleaning)")

plt.tight_layout()
plt.show()

## 5. Cleaning Pipeline Summary

Each text goes through these steps (see `data_clean.py` for implementation):

| Step | Action | Example |
|------|--------|---------|
| 1 | Decode HTML entities | `&amp;` → `&` |
| 2 | Remove URLs | `https://t.co/abc` → `` |
| 3 | Remove @mentions | `@prabowo` → `` |
| 4 | Strip `#` symbol | `#MBG` → `MBG` |
| 5 | Normalise smart quotes | `"..."` → `"..."` |
| 6 | Remove emojis | `🟣😂` → `` |
| 7 | Collapse whitespace | `hello   world` → `hello world` |
| 8 | Strip wrapping quotes | `"..."` → `...` |

### Filters applied after cleaning:
- Drop empty / whitespace-only rows
- Drop rows with fewer than **2 words** or fewer than **5 characters**
- Drop exact duplicate texts

## 6. Load Cleaned Dataset & Compare

In [ ]:
df_clean = pd.read_csv("../../dataset/cleaned/cleaned.csv")
print(f"Raw rows:    {len(df_raw)}")
print(f"Cleaned rows: {len(df_clean)}")
print(f"Removed:      {len(df_raw) - len(df_clean)} rows ({100 * (len(df_raw) - len(df_clean)) / len(df_raw):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

before = df_raw["source"].value_counts()
after = df_clean["source"].value_counts()

x = range(len(before))
width = 0.35
axes[0].bar([i - width/2 for i in x], before.values, width, label="Before", color="coral")
axes[0].bar([i + width/2 for i in x], after.values, width, label="After", color="steelblue")
axes[0].set_xticks(x)
axes[0].set_xticklabels(before.index)
axes[0].set_title("Row Count Before vs After Cleaning")
axes[0].legend()

removed_pct = 100 * (before - after) / before
axes[1].bar(removed_pct.index, removed_pct.values,
            color=[colors.get(s, "gray") for s in removed_pct.index])
axes[1].set_title("% Removed per Source")
axes[1].set_ylabel("%")
for i, v in enumerate(removed_pct.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 7. Word Cloud (Cleaned)

In [ ]:
all_text = " ".join(df_clean["text"].dropna().astype(str))
wc = WordCloud(
    width=800, height=400,
    background_color="white",
    colormap="viridis",
    max_words=100,
    collocations=False,
).generate(all_text)

plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Most Frequent Words in Cleaned MBG Dataset", fontsize=14)
plt.show()

## 8. Final Cleaned Dataset Stats

In [ ]:
print("=== Final Cleaned Dataset ===")
print(f"Total rows        : {len(df_clean)}")
print(f"Avg characters     : {df_clean['text'].str.len().mean():.0f}")
print(f"Avg words          : {df_clean['text'].str.split().str.len().mean():.0f}")
print(f"Min / Max words   : {df_clean['text'].str.split().str.len().min()} / {df_clean['text'].str.split().str.len().max()}")
print(f"Date range         : {df_clean['date'].min()} → {df_clean['date'].max()}")
print()

for src in ["x", "kaggle", "youtube", "instagram"]:
    subset = df_clean[df_clean["source"] == src]
    print(f"  {src:12s} : {len(subset):5d} rows | avg {subset['text'].str.split().str.len().mean():5.0f} words")

In [ ]:
df_clean.sample(10, random_state=42)[["source", "text"]]

---

## Next Steps

1. **Labelling** → annotate a subset with positive / negative / neutral labels
2. **Preprocessing** → tokenisation, stopword removal, stemming for SVM
3. **Training** → train SVM model with TF-IDF features
4. **Evaluation** → cross-validation, confusion matrix, classification report